# Sıfırdan Dil Modeli · Bölüm 9
## GPT'yi birleştirmek: ilk metin

Sekiz bölümdür parça topluyoruz: tokenizer, kayan pencere, gömme, dikkat, LayerNorm, GELU, ileri besleme, kısa yollar. Bu defterde hepsini birleştiriyoruz: önce bir **transformer bloğu**, sonra blokları üst üste koyan **GPTModel**. Parametrelerini sayıyor, sonra modele ilk metnini yazdırıyoruz. Eğitilmemiş model anlamsız konuşur; defterin sonunda aynı modele Çırak'ın eğitilmiş ağırlıklarını yüklüyoruz.

Adımlar Sebastian Raschka'nın *Build a Large Language Model (From Scratch)* kitabının 4.5–4.7. bölümlerini izler. Kod, kitabın açık kaynak deposundaki kodun ([rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Apache-2.0) Türkçe metin için uyarlamasıdır.

Hücreleri sırayla çalıştırın: hücreye tıklayın, **Shift + Enter**'a basın.

## 1. Önceki bölümlerin kodu
Dikkat, LayerNorm, GELU ve ileri besleme önceki bölümlerde yazdığımız parçalar; `onceki_bolumler.py` dosyasında duruyorlar (kitaptaki `previous_chapters.py`'nin karşılığı). Dosyayı ve Çırak'ın tokenizer'ını depodan indiriyoruz:

In [ ]:
import os
import urllib.request

repo = ("https://raw.githubusercontent.com/"
        "kocahmet1/sifirdan-dil-modeli/main/")
for path in ["onceki_bolumler.py", "veri/cirak_tokenizer.json"]:
    file_name = os.path.basename(path)
    if not os.path.exists(file_name):
        urllib.request.urlretrieve(repo + path, file_name)
    print(file_name, os.path.getsize(file_name), "bayt")

Parçaları dosyadan alıyoruz. `cirak_tokenizer()` Bölüm 2'deki BPE tokenizer'ımızı Çırak'ın birleştirmeleriyle kurar.

In [ ]:
import torch
import torch.nn as nn
from onceki_bolumler import (cirak_tokenizer, MultiHeadAttention,
                             LayerNorm, FeedForward, GPT_CONFIG)

tokenizer = cirak_tokenizer()
print(len(tokenizer.vocab), "parça")

Modelin ayarları da geçen bölümden: kitaptaki GPT-2 ayarları yerine Çırak'ın boyu. `items()` bir sözlüğün anahtar-değer çiftlerini gezer.

In [ ]:
for key, value in GPT_CONFIG.items():
    print(f"{key:15} {value}")

## 2. Transformer bloğu
Bir blok iki kısımdan oluşur: önce **dikkat** (tokenlar birbirine bakar), sonra **ileri besleme** (her token kendi vektörünü işler). İkisinden önce bir `LayerNorm`, ikisinin etrafında bir **kısa yol**: girdiyi sakla, işlemden geçir, girdiyi geri ekle. `nn.Module` PyTorch'taki her katmanın temel sınıfı; `forward` verinin katmandan nasıl geçtiğini söyler.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x                  # kısa yol: girdiyi sakla
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut              # girdiyi geri ekle

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

Bloğu deneyelim: 2 metin, her birinde 4 token, her token 256 sayı. `torch.rand` 0 ile 1 arasında rastgele sayılar üretir; `manual_seed` aynı rastgele sayıların her seferinde gelmesini sağlar.

In [ ]:
torch.manual_seed(123)
x = torch.rand(2, 4, 256)   # 2 metin · 4 token · 256 sayı
block = TransformerBlock(GPT_CONFIG)
output = block(x)

print("Girdi şekli:", x.shape)
print("Çıktı şekli:", output.shape)

## 3. GPT modeli
Girişte iki gömme tablosu (token ve konum), ortada `n_layers` tane blok, sonda bir `LayerNorm` ve çıkış katmanı. `nn.Sequential` blokları sırayla birbirine bağlar; köşeli parantezli satır altı bloğu bir listeye koyar, baştaki `*` listeyi açıp tek tek verir. Çıkış katmanı her tokenın 256 sayısını, sözlükteki 4.096 parçanın her birine bir puana çevirir: **logit**.

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

## 4. İlk çıktı
Modele iki kısa metin verelim. İkisi de 4 parça; `torch.stack` iki satırı tek bir tablo (batch) yapar.

In [ ]:
batch = []
txt1 = "Bir gün babam"
txt2 = "Bir gün Hasan"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG)

out = model(batch)
print("Çıktı şekli:", out.shape)
print(out)

## 5. Parametreler
Modelin öğrenecek sayılarına **parametre** denir. `numel` bir tensördeki sayı adedini verir; hepsini topluyoruz:

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre: {total_params:,}")

Gömme tablosu ile çıkış katmanı aynı şekilde: 4.096 × 256. GPT-2 (ve Çırak) ikisini tek bir tabloda paylaşır (*weight tying*). Paylaşılsaydı parametre sayısı:

In [ ]:
print("Gömme katmanı:", model.tok_emb.weight.shape)
print("Çıkış katmanı:", model.out_head.weight.shape)

total_params_tied = (total_params -
                     sum(p.numel() for p in model.out_head.parameters()))
print(f"Ağırlık paylaşımıyla: {total_params_tied:,}")

Bir blokta parametreler nerede? Dikkat mi, ileri besleme mi? (Kitabın 4.1. alıştırması)

In [ ]:
first_block = model.trf_blocks[0]
att_params = sum(p.numel() for p in first_block.att.parameters())
ff_params = sum(p.numel() for p in first_block.ff.parameters())
print(f"Bir blokta dikkat:        {att_params:,}")
print(f"Bir blokta ileri besleme: {ff_params:,}")

Her parametre bir `float32`: 4 bayt. Model bellekte ne kadar yer tutar?

In [ ]:
total_size_mb = total_params * 4 / (1024 * 1024)
print(f"Modelin boyutu: {total_size_mb:.2f} MB")

Aynı sınıf, kitaptaki GPT-2 ayarlarıyla (GPT-2'nin en küçüğü, 124M):

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, "context_length": 1024, "emb_dim": 768,
    "n_heads": 12, "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False
}
gpt2 = GPTModel(GPT_CONFIG_124M)
gpt2_params = sum(p.numel() for p in gpt2.parameters())
print(f"GPT-2 ayarlarıyla: {gpt2_params:,}")
print(f"Ağırlık paylaşımıyla: {gpt2_params - gpt2.out_head.weight.numel():,}")

## 6. Metin üretmek
Model her konum için sıradaki parçanın puanlarını verir. Metin üretmek için yalnız **son** konumun puanlarına bakarız: en olası parçayı seçer, metnin sonuna ekler, tekrarlarız. Önce başlangıç metnini sayılara çeviriyoruz; `unsqueeze(0)` başa bir batch boyutu ekler.

In [ ]:
start_context = "Bir gün babam"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

Tek bir adım: `eval()` dropout'u kapatır, `no_grad()` gradyan hesabını atlar (öğrenmiyoruz, yalnız tahmin ediyoruz). Son konumun puanlarını `softmax` ile olasılığa çeviriyoruz; `topk` en büyük beşini verir.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(encoded_tensor)

logits = logits[:, -1, :]                 # yalnız son tokenın puanları
probas = torch.softmax(logits, dim=-1)    # puanlar → olasılıklar
top = torch.topk(probas, 5)               # en olası beş parça
for p, i in zip(top.values[0], top.indices[0]):
    print(f"{p.item():.5f}", repr(tokenizer.decode([i.item()])))

Aynı adımı bir döngüde tekrarlayan fonksiyon (kitaptaki `generate_text_simple`). `argmax` en büyük olasılığın sırasını verir; `torch.cat` yeni parçayı metnin sonuna ekler.

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]     # en fazla context_size token
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]             # son tokenın puanları
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)   # sona ekle
    return idx

In [ ]:
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=10,
    context_size=GPT_CONFIG["context_length"]
)
print("Çıktı:", out)
print("Uzunluk:", len(out[0]))
print(tokenizer.decode(out.squeeze(0).tolist()))

## 7. Aynı model, Çırak'ın ağırlıklarıyla
Model henüz hiçbir şey öğrenmedi: sayıları rastgele. Serinin modeli Çırak, Türkçe Vikipedi'de eğitildi ve bizim `GPTModel`'imizle aynı boyda. Onun eğitilmiş sayılarını bizim modelin katman adlarıyla `cirak_gpt.pt` dosyasına kaydettik; `load_state_dict` sayıları modelin içine yerleştirir. (Çırak'ı eğiten kodla bizim sınıflarımız arasındaki eşlemeyi Bölüm 13'te kendimiz yazacağız.)

In [ ]:
file_name = "cirak_gpt.pt"
if not os.path.exists(file_name):
    urllib.request.urlretrieve(repo + "veri/" + file_name, file_name)
print(file_name, os.path.getsize(file_name) // 2**20, "MB")

print(model.load_state_dict(torch.load(file_name)))

In [ ]:
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=30,
    context_size=GPT_CONFIG["context_length"]
)
print(tokenizer.decode(out.squeeze(0).tolist()))

Çırak aynı başlangıçtan sonra hangi parçaları bekliyor?

In [ ]:
with torch.no_grad():
    logits = model(encoded_tensor)[:, -1, :]
probas = torch.softmax(logits, dim=-1)
top = torch.topk(probas, 5)
for p, i in zip(top.values[0], top.indices[0]):
    print(f"{p.item():.5f}", repr(tokenizer.decode([i.item()])))

## Özet
Bir transformer bloğu: LayerNorm → dikkat → kısa yol, LayerNorm → ileri besleme → kısa yol; girdinin şekli değişmez, bu yüzden bloklar üst üste konabilir. `GPTModel` gömmeleri, altı bloğu, son LayerNorm'u ve çıkış katmanını birleştirir: her token için 4.096 puan. `generate_text_simple` en olası parçayı seçip sona ekleyerek metin üretir. Eğitilmemiş model anlamsız konuşur; aynı kod Çırak'ın sayılarıyla Türkçe yazar, ama en olası parçayı seçmekte ısrar ettiği için kendini tekrarlar.

**Kendin dene**
- `GPT_CONFIG`'in bir kopyasında `n_layers`'ı 12, `emb_dim`'i 512 yapın: parametre sayısı kaç oluyor?
- Çırak'ın ağırlıklarıyla kendi başlangıç metninizi deneyin: model neyi tekrar ediyor?

**Sıradaki bölüm:** model ne kadar yanılıyor? Kayıp ve eğitimin ilk adımı.

---
Kaynak: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Manning 2024, bölüm 4.5–4.7. Kod: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache-2.0), Türkçe uyarlama: Sıfırdan Dil Modeli. Çırak'ın tokenizer'ı ve ağırlıkları Türkçe Vikipedi metniyle (CC BY-SA 4.0) eğitildi.